# Load Metadata

## Load the Video Segments generated in Step 1
From "Add Input" tab on the right, select the "Kulintang(1)-Video_Segmentation_silence_interval", and then run the next code block.

## Set the metadata csv path

In [1]:
CSV_PATH = '/kaggle/input/datasets/maryrobynbernabe/kulintang-videosegments/Kulingtang_metadata_updated.csv'

## Load the csv file as a dataframe

In [2]:
import pandas as pd
df = pd.read_csv(CSV_PATH)
df = df[df['label'] != 'N']
df.head()

,video_name,video_root_folder,audio_name,audio_root_folder,video_segment_name,video_segment_path,label
4,IMG_0037.MOV,/kaggle/input/datasets/maryrobynbernabe/kulint...,IMG_0037.wav,/kaggle/input/datasets/maryrobynbernabe/kulint...,IMG_0037_part5.mov,/kaggle/input/datasets/maryrobynbernabe/kulint...,2
5,IMG_0037.MOV,/kaggle/input/datasets/maryrobynbernabe/kulint...,IMG_0037.wav,/kaggle/input/datasets/maryrobynbernabe/kulint...,IMG_0037_part6.mov,/kaggle/input/datasets/maryrobynbernabe/kulint...,3
6,IMG_0037.MOV,/kaggle/input/datasets/maryrobynbernabe/kulint...,IMG_0037.wav,/kaggle/input/datasets/maryrobynbernabe/kulint...,IMG_0037_part7.mov,/kaggle/input/datasets/maryrobynbernabe/kulint...,4
7,IMG_0037.MOV,/kaggle/input/datasets/maryrobynbernabe/kulint...,IMG_0037.wav,/kaggle/input/datasets/maryrobynbernabe/kulint...,IMG_0037_part8.mov,/kaggle/input/datasets/maryrobynbernabe/kulint...,5
11,IMG_0037.MOV,/kaggle/input/datasets/maryrobynbernabe/kulint...,IMG_0037.wav,/kaggle/input/datasets/maryrobynbernabe/kulint...,IMG_0037_part12.mov,/kaggle/input/datasets/maryrobynbernabe/kulint...,8


# Getting all the video segments, extracting audio, converting to features and saving to csv

## Required installations

In [3]:
!pip install moviepy librosa matplotlib numpy -q

## Function to extract audio

In [4]:
from moviepy.editor import VideoFileClip
import os

# Create the temp directory if it doesn't exist
os.makedirs('/kaggle/temp', exist_ok=True)

def extract_audio(video_path):
    """
    Extracts the audio from a video file and saves it as a WAV file.
    """
    video = VideoFileClip(video_path)
    # Ensure this matches the path used in your feature extraction loop
    audio_output_path = "/kaggle/temp/audio.wav"
    
    audio = video.audio
    # Use logger=None to keep the output clean during loops
    audio.write_audiofile(audio_output_path, logger=None)
    
    # Close the video reader to free up memory and system resources
    video.close()

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
error: XDG_RUNTIME_DIR not set in the environment.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
A

## Function to extract audio features

In [5]:
import librosa
import pandas as pd
import numpy as np

#function to extract features from audion files
def extract_features(audio_path):
    """
    Extracts various audio features from an audio file.
    :param audio_path: The path to the audio file.
    :return: A dictionary containing the extracted features.
    """

    # load the audio file
    y,sr = librosa.load(audio_path,mono=True) #load the audio file
    # extract features
    rmse = librosa.feature.rms(y=y)[0] #compute root-mean-square (RMS) value for each frame
    spec_cent = librosa.feature.spectral_centroid(y=y,sr=sr) #calculate spectral_centroid
    spec_bw = librosa.feature.spectral_bandwidth(y=y,sr=sr) #calculate spectral_bandwith
    rolloff = librosa.feature.spectral_rolloff(y=y,sr=sr) #calculate spectral_rolloff
    zcr = librosa.feature.zero_crossing_rate(y) #calculate zero crossing rate
    mfcc = librosa.feature.mfcc(y=y,sr=sr) #Mel frequency ceptral coefficients (mfcc)
    
    audio_dict={
    'RMSE':rmse.mean(),
    'SPECTRAL_CENTROID':spec_cent.mean(),
    'SPECTRAL_BANDWIDTH':spec_bw.mean(),
    'ROLLOFF':rolloff.mean(),
    'ZERO_CROSSING_RATE':zcr.mean()
    }
    
    #add the mfcc values
    for index,mfcc_set in enumerate(mfcc):
        feature_name=f"MFCC_FEATURE_{index}"
        audio_dict[feature_name]=np.mean(mfcc_set)
    
    return audio_dict

## Extracting audio features from each segment and save to a dataframe

In [6]:
from tqdm.notebook import tqdm

audio_features = []
# This must match the path inside extract_audio
audio_file_path = "/kaggle/temp/audio.wav"

for file_path in tqdm(df['video_segment_path'], desc="Processing Files"):
    if os.path.exists(file_path):
        extract_audio(file_path)
        features = extract_features(audio_file_path)
        audio_features.append(features)
    else:
        print(f"Warning: File not found at {file_path}")

features_df = pd.DataFrame(audio_features)
df_new = pd.concat([df["video_segment_name"].reset_index(drop=True), features_df], axis=1)

Processing Files:   0%|          | 0/1081 [00:00<?, ?it/s]

## Reset the dataframe index

In [7]:
df_label = df["label"]
df_label = df_label.reset_index(drop=True)
print(df_label)

0       2
1       3
2       4
3       5
4       8
       ..
1076    1
1077    2
1078    3
1079    6
1080    7
Name: label, Length: 1081, dtype: object


## Attaching the label to dataframe

In [8]:
# Concatenate the DataFrames
df_main = pd.concat([df_new, df_label], axis=1)

df_main.head()

,video_segment_name,RMSE,SPECTRAL_CENTROID,SPECTRAL_BANDWIDTH,ROLLOFF,ZERO_CROSSING_RATE,MFCC_FEATURE_0,MFCC_FEATURE_1,MFCC_FEATURE_2,MFCC_FEATURE_3,...,MFCC_FEATURE_11,MFCC_FEATURE_12,MFCC_FEATURE_13,MFCC_FEATURE_14,MFCC_FEATURE_15,MFCC_FEATURE_16,MFCC_FEATURE_17,MFCC_FEATURE_18,MFCC_FEATURE_19,label
0,IMG_0037_part5.mov,0.110914,636.822926,730.140212,704.956055,0.042457,-388.798676,127.769524,30.357569,0.964475,...,-9.094934,2.113101,9.804668,13.554678,10.162496,5.915236,7.076756,9.991294,4.527101,2
1,IMG_0037_part6.mov,0.105051,802.982334,848.353061,964.883256,0.048331,-370.882080,147.863907,10.282375,-6.605200,...,-0.436702,17.252119,17.207047,9.353372,2.422301,2.440642,4.783227,-7.288693,-19.787868,3
2,IMG_0037_part7.mov,0.117851,824.580504,817.884068,992.065430,0.050767,-356.948303,154.975510,5.095808,-17.982309,...,11.097885,14.794455,7.392744,-1.549907,-4.859143,-3.377399,-8.645940,-23.538469,-19.215622,4
3,IMG_0037_part8.mov,0.070839,1288.281722,1206.854531,1982.711088,0.076848,-268.320526,161.854477,-57.438450,-28.290649,...,14.017271,-4.714745,-6.482290,1.976947,-0.569431,-14.925186,-21.212166,-12.418090,-4.545398,5
4,IMG_0037_part12.mov,0.098182,1416.618909,981.970283,1715.254211,0.091492,-315.661072,115.525940,-67.717110,-48.924007,...,-9.691249,7.581002,0.971637,0.144251,6.229625,-5.574753,-19.128342,-13.515326,9.690474,8


## Check for duplicate entries

In [9]:
# Check for duplicate values
df_main.duplicated().sum()

np.int64(0)

## Check for empty cells

In [10]:
# Check for null values
df_main.isnull().sum()

video_segment_name    0
RMSE                  0
SPECTRAL_CENTROID     0
SPECTRAL_BANDWIDTH    0
ROLLOFF               0
ZERO_CROSSING_RATE    0
MFCC_FEATURE_0        0
MFCC_FEATURE_1        0
MFCC_FEATURE_2        0
MFCC_FEATURE_3        0
MFCC_FEATURE_4        0
MFCC_FEATURE_5        0
MFCC_FEATURE_6        0
MFCC_FEATURE_7        0
MFCC_FEATURE_8        0
MFCC_FEATURE_9        0
MFCC_FEATURE_10       0
MFCC_FEATURE_11       0
MFCC_FEATURE_12       0
MFCC_FEATURE_13       0
MFCC_FEATURE_14       0
MFCC_FEATURE_15       0
MFCC_FEATURE_16       0
MFCC_FEATURE_17       0
MFCC_FEATURE_18       0
MFCC_FEATURE_19       0
label                 0
dtype: int64

## Save the features to a csv file

In [11]:
# Corrected save path with leading slash
df_main.to_csv('/kaggle/working/featurizedaudio.csv', index=False)